# 02 - Experiments

Unsupervised vs supervised models, SMOTE vs no SMOTE, recall at fixed false-positive budgets, and random vs time-based evaluation.
All model choices are made on a **validation split taken from the training data**, so the final test set stays untouched. The full pipeline is `python -m src.train`.

The hyper-parameters below are the ones the final search selected (`models/metrics.json`), not re-tuned here.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

from src.data import SEED, build_features, load_data
from src.evaluate import operating_point, recall_at_fpr
from src.models import SmoteXGB, iforest_scores, threshold_for_fpr

best = json.load(open("../models/metrics.json"))["best_params"]
df = load_data("../data/creditcard.csv")
X, y, iso_cols = build_features(df)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
# carve a validation split out of the training data; X_te is not used in this notebook
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.25, stratify=y_tr, random_state=SEED)
print(len(X_fit), int(y_fit.sum()), "fraud |", len(X_val), int(y_val.sum()), "fraud")

## Unsupervised vs supervised, SMOTE vs no SMOTE

In [ ]:
oof, iso_val, _ = iforest_scores(X_fit[iso_cols].values, X_val[iso_cols].values)
Xh_fit = X_fit.assign(iforest_score=oof)
Xh_val = X_val.assign(iforest_score=iso_val)

scores = {
    "A. Isolation Forest only": iso_val,
    "B. XGBoost + SMOTE (no IF)": SmoteXGB(use_smote=True, **best).fit(X_fit, y_fit).predict_proba(X_val)[:, 1],
    "C. IF + XGBoost, no SMOTE": SmoteXGB(use_smote=False, **best).fit(Xh_fit, y_fit).predict_proba(Xh_val)[:, 1],
    "D. Hybrid IF + XGBoost + SMOTE": SmoteXGB(use_smote=True, **best).fit(Xh_fit, y_fit).predict_proba(Xh_val)[:, 1],
}
rows = []
for name, p in scores.items():
    rows.append({"model": name, "ROC-AUC": roc_auc_score(y_val, p), "PR-AUC": average_precision_score(y_val, p),
                 "recall @0.1% FPR": recall_at_fpr(y_val, p, 0.001)})
pd.DataFrame(rows).set_index("model").round(3)

With so few fraud cases in a validation split, differences of one or two caught frauds move recall by a point or more. Read small gaps as noise.

## Random vs time-based split

The main pipeline uses a random split. A time-based split (train on the earlier part of the two days, evaluate on the later part) is closer to how a fraud model is used. The window is short and has few fraud cases, so this is only a sanity check.

In [ ]:
order = df["Time"].argsort().values
cut = int(0.8 * len(df))
tr_idx, te_idx = order[:cut], order[cut:]
X_t, y_t = X.iloc[tr_idx], y.iloc[tr_idx]
X_f, y_f = X.iloc[te_idx], y.iloc[te_idx]
print("train fraud:", int(y_t.sum()), "| later-window fraud:", int(y_f.sum()))

model = SmoteXGB(use_smote=True, **best).fit(X_t, y_t)
p = model.predict_proba(X_f)[:, 1]
print("time-based  PR-AUC:", round(average_precision_score(y_f, p), 3), "| ROC-AUC:", round(roc_auc_score(y_f, p), 3))

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
pr = SmoteXGB(use_smote=True, **best).fit(Xr_tr, yr_tr).predict_proba(Xr_te)[:, 1]
print("random split PR-AUC:", round(average_precision_score(yr_te, pr), 3), "| ROC-AUC:", round(roc_auc_score(yr_te, pr), 3))

## Feature importance of the final model

The components are PCA outputs, so they cannot be interpreted as business features. The anomaly score is ranked among the weaker features.

In [ ]:
import joblib
bundle = joblib.load("../models/fraud_model.joblib")
imp = pd.Series(bundle["hybrid"].feature_importances_, index=bundle["feature_names"]).sort_values(ascending=False)
imp.head(8).round(3), imp.rank(ascending=False)["iforest_score"]